
# 02 — Speculative Decoding, from Scratch

**Goal:** implement the accept/reject core of speculative decoding, prove to yourself (via
simulation) that it produces *exactly* the target model's distribution, and be able to derive
the speedup math and discuss variants (tree-based, EAGLE/hidden-state drafting, multimodal
drafting) fluently in an interview. This is presumably close to your own area of work, so the
quiz leans a little harder here than in the other notebooks.

Structure: **Lesson → Implementation → Quiz → Final Answers & Explanations.**



## 1. Lesson

### 1.1 The bottleneck speculative decoding targets

Autoregressive LM decoding generates one token per forward pass. On modern accelerators, a
single forward pass over one new token is **memory-bandwidth bound**, not compute bound: you
must stream the entire model's weights (and the KV cache) through the chip to produce a single
token, so the GPU's compute units sit mostly idle. This means running the model on 1 token or on
5 tokens *in parallel* costs roughly the *same* wall-clock time — the bottleneck is moving
weights, not doing the matmuls. Speculative decoding exploits exactly this: verifying $k$
speculatively-drafted tokens in one parallel forward pass of the (expensive) target model costs
about the same as generating a single token normally would, but *produces up to $k+1$ tokens* if
the draft was good.

### 1.2 The algorithm

At each step, given a prefix, and a small **draft model** $q$ that's cheap to run:

1. **Draft**: autoregressively sample $k$ tokens $\hat{x}_1, \dots, \hat{x}_k$ from the draft
   model, recording $q(\hat{x}_i \mid \text{prefix}, \hat{x}_{<i})$ for each.
2. **Verify**: run the (expensive) **target model** $p$ once, in parallel, over the prefix plus
   all $k$ drafted tokens, obtaining $p(\cdot \mid \text{prefix}, \hat{x}_{<i})$ for every
   position $i = 1, \dots, k$ (this is a *single* forward pass because the target model can
   process $k$ new positions in parallel, unlike autoregressive generation).
3. **Accept/reject**, token by token, from $i=1$: accept $\hat{x}_i$ with probability
   $$\min\left(1, \frac{p(\hat{x}_i \mid \cdot)}{q(\hat{x}_i \mid \cdot)}\right).$$
   - If accepted, move to $i+1$ and repeat.
   - If **rejected** (which happens with the complementary probability), **stop** drafting
     acceptance at this position, and instead sample **one replacement token** from the
     residual distribution
     $$
     p'(x) = \text{normalize}\big(\max(0,\, p(x) - q(x))\big)
     $$
     i.e. whatever probability mass $p$ places on $x$ beyond what $q$ already accounted for,
     renormalized to sum to 1. This replacement token is appended and the rest of the drafted
     tokens (from $i+1$ onward) are discarded.
4. If **all $k$** drafted tokens are accepted, sample one **bonus token** directly from
   $p(\cdot \mid \text{prefix}, \hat{x}_{1:k})$ (free — the target model already computed this
   distribution during verification) and append it too. So a fully-successful step yields
   $k+1$ new tokens for the cost of one target forward pass.

### 1.3 Why this is *exact*, not an approximation

This is the part interviewers most often probe. Claim: for any fixed prefix, the marginal
distribution of the token this procedure outputs at each position is *exactly* $p$ (the target
model's distribution) — never $q$, and never some blend.

Proof sketch: consider a single position with proposal distribution $q$ and target $p$. The
probability that the procedure outputs a specific token $x$ is

$$
P(\text{output} = x) = q(x)\cdot\min\!\left(1, \frac{p(x)}{q(x)}\right) \;+\; P(\text{reject})\cdot p'(x)
$$

The first term is "$x$ was proposed by $q$ *and* accepted." The second term is "*something* was
rejected, and the residual sampling step happened to pick $x$." Expand
$q(x)\min(1, p(x)/q(x)) = \min(q(x), p(x))$, and $P(\text{reject}) = \sum_y \max(0, p(y)-q(y)) = 1 - \sum_y \min(p(y),q(y))$,
and $p'(x) = \max(0, p(x)-q(x)) / P(\text{reject})$. Substituting:

$$
P(\text{output}=x) = \min(p(x), q(x)) + \max(0, p(x) - q(x)) = p(x)
$$

(the last equality holds because for any $x$, either $p(x)\le q(x)$, giving $p(x) + 0 = p(x)$,
or $p(x) > q(x)$, giving $q(x) + (p(x)-q(x)) = p(x)$). So regardless of how good or bad the draft
model is, **the output distribution is always exactly $p$** — a bad draft model only costs you
*speed* (more rejections → more expensive target-only fallback steps), never *correctness*.

### 1.4 Expected speedup

Let $\alpha$ = expected per-token acceptance probability (averaged appropriately over the draft
distribution, assumed roughly constant across positions for a back-of-envelope estimate). In one
speculative round with draft length $k$, the expected number of tokens produced is

$$
\mathbb{E}[\text{tokens per round}] = \frac{1 - \alpha^{k+1}}{1 - \alpha}
$$

(a geometric-series sum: probability of accepting at least $i$ tokens is $\alpha^i$, summed over
$i=0,\dots,k$, plus handling of the bonus token — this is the standard result from the
speculative decoding papers). Each round costs roughly one target forward pass (the expensive
part) plus $k$ cheap draft forward passes. As $\alpha \to 1$ (near-perfect draft), expected
tokens/round $\to k+1$; as $\alpha \to 0$, it $\to 1$ (you get no speedup, every step degenerates
to a single target-model token, same as ordinary decoding but *slightly worse* due to wasted
draft compute). This immediately tells you why $k$ can't be pushed arbitrarily high for free:
past some point, the marginal expected tokens gained per additional draft step (which costs real
wall-clock time to draft, and a wider verify pass) shrinks like $\alpha^k$, while the drafting
cost grows linearly with $k$ — there's an optimal $k$ that depends on $\alpha$ and the
target/draft cost ratio.

### 1.5 Variants worth knowing

- **Tree-based / SpecInfer / Medusa**: instead of one single chain of $k$ draft tokens, draft a
  *tree* of candidate continuations (e.g. top-few tokens at each position, branching), then
  verify the whole tree in one target forward pass using a tree-structured attention mask.
  Increases the chance *some* path is accepted at each depth, at the cost of a wider (but still
  single) verification pass and a more complex accept/reject procedure over tree paths.
- **EAGLE-style hidden-state drafting**: rather than a fully separate small LM, draft using a
  lightweight head that consumes the *target model's own hidden states* (not just its output
  tokens) to predict future tokens — hidden states carry much more signal than a token stream
  alone, giving a cheap draft head much higher acceptance rates for its size.
- **Multimodal speculative decoding**: when the target is a VLM, the draft model ideally also
  needs to be conditioned on the image, but a full second vision encoder + small LM stack is
  expensive. Reusing the target VLM's own vision features (or hidden states, à la EAGLE) so the
  draft head is cheap yet still visually grounded — rather than drafting text "blind" to the
  image — is precisely the kind of problem hidden-state-distillation draft models (e.g. ViSpec/
  HiViS-style approaches) are built to solve, and is a natural thing to bring up if asked how
  speculative decoding changes for VLMs versus text-only LLMs.


In [ ]:

import numpy as np
rng = np.random.default_rng(0)



## 2. Implementation

We'll work with toy categorical distributions over a small vocabulary (no real models needed —
the accept/reject logic is exactly the same regardless of vocab size or how $p, q$ are produced).
Fill in every `# TODO`.


In [ ]:

def normalize(p):
    p = np.clip(p, 0, None)
    s = p.sum()
    if s <= 1e-12:
        # degenerate case: p and q identical everywhere -> should never need this branch
        return np.ones_like(p) / len(p)
    return p / s


def sample_categorical(probs, rng):
    return rng.choice(len(probs), p=probs)


def speculative_step(target_probs_fn, draft_probs_fn, prefix, k, rng):
    \"\"\"
    One round of speculative decoding.

    target_probs_fn(prefix) -> np.ndarray of shape (vocab,), the TARGET model's next-token
        distribution given `prefix` (a list of ints). You may assume it's cheap here (toy setting);
        in reality this would be one expensive forward pass evaluated at all draft positions at once.
    draft_probs_fn(prefix) -> np.ndarray of shape (vocab,), the DRAFT model's next-token distribution.

    Returns: (new_tokens: list[int], num_accepted: int)
        new_tokens are the tokens to append to prefix (drafted+accepted, plus exactly one more
        token from either a rejection-residual sample or a bonus sample).
        num_accepted is how many of the k drafted tokens were accepted (0..k).
    \"\"\"
    draft_tokens = []
    draft_dists = []
    cur_prefix = list(prefix)

    # --- Step 1: draft k tokens autoregressively from the draft model ---
    for _ in range(k):
        # TODO: get q = draft_probs_fn(cur_prefix), sample a token from it,
        # append token to draft_tokens, append q to draft_dists, extend cur_prefix
        pass

    # --- Step 2: "verify" -- get the target distribution at every drafted position ---
    # In a real system this is ONE parallel forward pass; here we just call it k+1 times
    # (once per position, including the final bonus/residual position) since this is a
    # numpy simulation, not a real batched model call.
    target_dists = []
    cur_prefix2 = list(prefix)
    for i in range(k):
        # TODO: append target_probs_fn(cur_prefix2) to target_dists, then extend cur_prefix2
        # with draft_tokens[i] (so the next target dist is conditioned on it, matching what
        # the drafted token assumed)
        pass

    # --- Step 3: accept/reject sequentially ---
    accepted = []
    num_accepted = 0
    for i in range(k):
        x_hat = draft_tokens[i]
        p = target_dists[i]
        q = draft_dists[i]
        # TODO: compute acceptance probability min(1, p[x_hat]/q[x_hat]) and flip a coin
        # (use rng.random() < accept_prob). If accepted: append x_hat to accepted, increment
        # num_accepted, continue loop. If rejected: sample replacement from normalize(max(0,p-q)),
        # append it to accepted, and `break` out of the loop (rest of draft is discarded).
        pass
    else:
        # for/else: only runs if the loop completed WITHOUT break, i.e. all k accepted
        # TODO: sample one bonus token from target_probs_fn(cur_prefix2) (the target dist
        # AFTER all k drafted tokens -- this is target_dists computed one step further; for
        # simplicity just call target_probs_fn again on the full accepted prefix) and append it
        pass

    return accepted, num_accepted


In [ ]:

def make_toy_distributions(vocab_size, rng, draft_quality=0.7, num_contexts=20):
    \"\"\"
    Returns (target_probs_fn, draft_probs_fn). Both are stateless functions of `prefix` here
    (toy setting: distribution depends only on len(prefix) mod num_contexts, just to make it
    deterministic and reproducible without a real model). `draft_quality` in [0,1] controls how
    close the draft distribution is to the target (1.0 = identical, 0.0 = unrelated random dist).
    Use num_contexts=1 to get a *stationary* toy process (acceptance rate alpha truly constant
    across positions) -- useful when you want to check the closed-form speedup formula cleanly,
    since that formula assumes a constant per-token alpha.
    \"\"\"
    base_target = rng.dirichlet(np.ones(vocab_size) * 0.5, size=num_contexts)
    noise = rng.dirichlet(np.ones(vocab_size) * 0.5, size=num_contexts)

    def target_probs_fn(prefix):
        ctx = len(prefix) % num_contexts
        return base_target[ctx]

    def draft_probs_fn(prefix):
        ctx = len(prefix) % num_contexts
        return normalize(draft_quality * base_target[ctx] + (1 - draft_quality) * noise[ctx])

    return target_probs_fn, draft_probs_fn



### Test A — statistical correctness

The core theoretical claim (section 1.3) is that speculative decoding's output at each position
is distributed *exactly* as the target model, regardless of draft quality. We check this by
running many independent single-token speculative rounds and comparing the empirical output
histogram against the target distribution directly, for both a good and a deliberately bad draft
model. If your implementation is correct, both should match the target distribution (bad drafts
should only make you reject more often — never bias the output).


In [ ]:

def chi_square_stat(counts, expected_probs, n):
    expected_counts = expected_probs * n
    return np.sum((counts - expected_counts) ** 2 / np.clip(expected_counts, 1e-9, None))


vocab_size = 6
n_trials = 20000

for draft_quality, label in [(0.9, "good draft"), (0.1, "bad draft")]:
    rng_local = np.random.default_rng(42)
    target_probs_fn, draft_probs_fn = make_toy_distributions(vocab_size, rng_local, draft_quality)
    prefix = []  # context index 0
    true_probs = target_probs_fn(prefix)

    outputs = []
    accept_count = 0
    for _ in range(n_trials):
        new_tokens, num_accepted = speculative_step(target_probs_fn, draft_probs_fn, prefix, k=1, rng=rng_local)
        outputs.append(new_tokens[0])
        accept_count += num_accepted

    counts = np.bincount(outputs, minlength=vocab_size)
    empirical = counts / n_trials
    chi2 = chi_square_stat(counts, true_probs, n_trials)
    # dof = vocab_size - 1 = 5; chi2 critical value at p=0.01 is ~15.09
    print(f"[{label}] acceptance rate={accept_count/n_trials:.3f}  chi2={chi2:.2f} "
          f"(should be well under ~15 for a good fit at alpha=0.01)")
    print(f"  target probs:    {np.round(true_probs, 3)}")
    print(f"  empirical probs: {np.round(empirical, 3)}")
    assert chi2 < 25, "empirical distribution diverges too much from target -- check accept/reject logic"

print("\nTest A passed: output distribution matches the target model regardless of draft quality.")



### Test B — speedup scales with acceptance rate as predicted

Check the $\mathbb{E}[\text{tokens/round}] = \frac{1-\alpha^{k+1}}{1-\alpha}$ formula empirically
by measuring average tokens produced per round across many rounds, for a few values of $k$, and
comparing to the closed form using the measured single-token acceptance rate $\alpha$. We use a
*stationary* toy process here (`num_contexts=1`, i.e. the same distribution at every position) so
that $\alpha$ is genuinely constant across positions -- exactly the assumption the closed-form
formula makes. (In Test A above, contexts varied with position, which is fine for checking
distributional correctness but would make $\alpha$ non-constant and only loosely match the
formula here.)


In [ ]:

def expected_tokens_per_round(alpha, k):
    if abs(alpha - 1.0) < 1e-9:
        return k + 1
    return (1 - alpha ** (k + 1)) / (1 - alpha)


rng_local = np.random.default_rng(7)
# num_contexts=1 -> a stationary toy process, so alpha is truly constant across positions
# and the closed-form formula should match the simulation tightly (not just approximately).
target_probs_fn, draft_probs_fn = make_toy_distributions(vocab_size, rng_local, draft_quality=0.85, num_contexts=1)

# first, measure single-token acceptance rate alpha directly
n_alpha_trials = 5000
accepts = 0
for _ in range(n_alpha_trials):
    _, num_accepted = speculative_step(target_probs_fn, draft_probs_fn, [], k=1, rng=rng_local)
    accepts += num_accepted
alpha = accepts / n_alpha_trials
print(f"measured alpha (single-token acceptance rate): {alpha:.3f}\n")

for k in [1, 2, 4, 8]:
    n_rounds = 3000
    total_tokens = 0
    for _ in range(n_rounds):
        new_tokens, _ = speculative_step(target_probs_fn, draft_probs_fn, [], k=k, rng=rng_local)
        total_tokens += len(new_tokens)
    empirical_avg = total_tokens / n_rounds
    predicted = expected_tokens_per_round(alpha, k)
    print(f"k={k}: empirical avg tokens/round={empirical_avg:.2f}, "
          f"formula prediction={predicted:.2f}")



## 3. Quiz

1. Derive (or re-derive from memory) why $P(\text{output}=x) = p(x)$ exactly, for any draft
   distribution $q$ with the same support. What breaks if $q(x) = 0$ somewhere that $p(x) > 0$?
2. What determines the expected number of accepted tokens per round, and concretely, why does
   pushing $k$ very high stop paying off?
3. Why is speculative decoding fundamentally a *memory-bandwidth* optimization rather than a
   FLOPs optimization? What would happen to its benefit on a hypothetical compute-bound (not
   bandwidth-bound) accelerator?
4. Compare tree-based speculative decoding to single-chain: what do you gain, and what do you
   pay for it?
5. In multimodal speculative decoding, why is it harder to build a good cheap draft model than
   in text-only speculative decoding? What's the appeal of drafting from the target VLM's own
   hidden states / vision features rather than training a fully independent small draft model?
6. Suppose your draft model is *systematically overconfident* (assigns higher probability to its
   top token than the target model would) versus *systematically underconfident*. Which hurts
   acceptance rate more, and why?
7. Why must the target model's verification forward pass process all $k$ drafted positions
   *in parallel* (not sequentially) for speculative decoding to provide a wall-clock speedup?

*(Your answers here)*



## 4. Final Answers & Explanations

### Q1 — Exactness derivation
See the derivation in section 1.3:
$P(\text{output}=x) = \min(p(x),q(x)) + \max(0, p(x)-q(x)) = p(x)$ for every $x$, because these
two terms are complementary depending on whether $p(x) \le q(x)$ or $p(x) > q(x)$. This holds
regardless of how bad $q$ is — a bad draft only changes how often you fall into the second
(rejection) term, never the *result* of the combined process. If $q(x) = 0$ for some $x$ with
$p(x) > 0$: the draft model can *never propose* $x$ directly, but the residual/rejection branch
still recovers it correctly — $\max(0, p(x) - q(x)) = p(x)$ in that case, so whenever a rejection
happens, $x$ gets its correct share of probability mass through the residual sampling step. The
formula is robust to $q$ having zero mass on some tokens; it would only break (divide-by-zero in
the acceptance ratio) if $q(x) = 0$ *and* the draft model somehow still proposed $x$, which can't
happen if you sample $\hat{x}$ from $q$ in the first place.

### Q2 — Diminishing returns on $k$
Expected tokens per round is $\frac{1-\alpha^{k+1}}{1-\alpha}$, which saturates toward
$\frac{1}{1-\alpha}$ as $k \to \infty$ (a finite ceiling, not infinity) — because the probability
that *all* $k$ drafted tokens are simultaneously accepted decays geometrically as $\alpha^k$.
Each additional unit of $k$ contributes an expected $\alpha^k$ extra accepted tokens, which
shrinks fast for any $\alpha < 1$, while the cost of drafting (k cheap forward passes, run
sequentially by the draft model) and the width of the verification pass both grow with $k$ —
past the point where $\alpha^k$ is small, you're paying real wall-clock cost for draft steps that
almost never get used.

### Q3 — Memory-bandwidth, not FLOPs
A single autoregressive decode step is memory-bandwidth bound: nearly all its wall-clock time is
spent streaming model weights (and KV cache) from HBM to compute units, while the actual matmul
FLOPs for one token are tiny relative to the hardware's peak compute — the compute units are
mostly idle, waiting on memory traffic. Because of this, processing $k{+}1$ tokens' worth of
matmuls in one pass costs almost the *same* wall-clock time as processing 1 token, since the
weight-streaming cost dominates and is paid either way — extra compute is nearly free. This is
exactly what a single parallel verification pass exploits: get up to $k{+}1$ tokens for
essentially the cost of one. On a hypothetical *compute-bound* accelerator (or with a large
enough batch size that you're no longer bandwidth-bound), this "compute is free" assumption
breaks down — verifying $k$ tokens in parallel would cost meaningfully more than verifying 1, and
speculative decoding's speedup would shrink or vanish, since you'd be paying close to full price
for the extra draft positions rather than getting them nearly free.

### Q4 — Tree-based vs. single-chain
Tree-based drafting (Medusa, SpecInfer) drafts multiple candidate continuations at each depth
(a branching tree) instead of one linear chain, then verifies the *entire tree* in a single
target forward pass using a tree-structured attention mask (so each path only attends to its own
ancestors). Gain: much higher probability that *some* path through the tree survives
verification for many steps, since you're no longer betting everything on one single greedy/
sampled chain — this raises the effective acceptance length substantially. Cost: the
verification pass processes more tokens per round (all tree nodes, most of which will be
discarded), the accept/reject bookkeeping is more complex (choosing which path to commit to),
and the attention mask needs tree-structured masking support, adding implementation complexity
versus the simple causal mask a single chain needs.

### Q5 — Multimodal drafting difficulty
In text-only speculative decoding, a small independent LM can be a perfectly reasonable draft
model because next-token prediction over text alone is a well-studied, learnable task at small
scale. In the multimodal case, the draft model's predictions need to be *grounded in the image*
too — a draft model that ignores the image (or uses a small, separately-trained vision encoder)
will systematically diverge from what the (image-conditioned) target VLM would say, tanking
acceptance rate; but giving the draft model a full independent vision encoder defeats the purpose
of making it cheap. The appeal of drafting from the target VLM's own hidden states / vision
features (EAGLE-style, or hidden-state-distillation approaches like ViSpec/HiViS) is that you get
the image-grounding *for free* — the expensive vision encoding was already computed once by the
target model — so the draft head only needs to be a small predictor consuming already-rich,
already-image-aware hidden states, rather than an entire second multimodal stack.

### Q6 — Overconfident vs. underconfident draft
An **overconfident** draft model (puts too much mass on its own top token relative to the
target) hurts more: acceptance probability is $\min(1, p(x)/q(x))$, and overconfidence means
$q(\hat x)$ is inflated for whatever token the draft actually proposes, directly shrinking
$p(\hat x)/q(\hat x)$ below 1 more often — i.e. you frequently propose a token the draft was
"too sure of" but the target likes noticeably less, causing rejection. An
**underconfident** draft (correct ranking, just flatter/more spread-out probabilities) tends to
still propose the right tokens about as often via sampling, and when it does, $q(\hat x)$ being
smaller than it "should" be only *helps* the ratio $p(\hat x)/q(\hat x)$ (it's more likely to
exceed 1, auto-accepting). So miscalibration in the direction of overconfidence is the more
damaging failure mode for acceptance rate; underconfidence mostly costs you a bit of proposal
diversity/quality rather than directly inflating rejections.

### Q7 — Why parallel verification is required
If the target model verified the $k$ drafted positions *sequentially* (one forward pass per
position, each depending on the previous), you'd pay the same per-token wall-clock cost as
ordinary autoregressive decoding — $k$ separate expensive passes for $k$ tokens, no better than
just generating with the target model directly (arguably worse, since you'd *also* have paid for
the draft model's passes). The entire speedup comes from the fact that a transformer can process
multiple new *positions* in a single forward pass by treating them as a batch along the sequence
dimension (using an appropriate causal/tree mask so each position only attends to its real
causal ancestors) — and because decoding is memory-bandwidth bound (Q3), that one wider pass
costs barely more than a single-token pass. Sequential verification would throw away exactly the
mechanism that makes any of this cheaper than plain autoregressive decoding.
